<a href="https://colab.research.google.com/github/swagata-dsai/loan-approval-prediction-flask/blob/main/Loan_Approval_Prediction_Using_ML_and_Flask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask scikit-learn pandas pyngrok -q

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

print("Uploaded file:", filename)

Saving train_u6lujuX_CVtuZ9i (1).csv.xls to train_u6lujuX_CVtuZ9i (1).csv (1).xls
Uploaded file: train_u6lujuX_CVtuZ9i (1).csv (1).xls


In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("File Name:", filename)
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Saving train_u6lujuX_CVtuZ9i (1).csv.xls to train_u6lujuX_CVtuZ9i (1).csv (2).xls
File Name: train_u6lujuX_CVtuZ9i (1).csv (2).xls
Dataset Shape: (614, 13)
Columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Prepare dataset
df = df.drop(columns=["Loan_ID"], errors="ignore")
df = df.dropna(subset=["Loan_Status"])

X = df.drop(columns=["Loan_Status"])
y = df["Loan_Status"]

# Identify column types
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Create model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train model
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Training Completed Successfully!")
print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Training Completed Successfully!
Model Accuracy: 75.61 %


In [ ]:
from flask import Flask, request
from pyngrok import ngrok
from getpass import getpass
from threading import Thread
import time
import pandas as pd

app = Flask("loan_prediction")

@app.route("/")
def home():
    return """
    <html>
    <body style="font-family:Arial; margin:40px;">
        <h2>Loan Approval Prediction System</h2>

        <form action="/predict" method="post">
            Gender:
            <select name="Gender">
                <option>Male</option>
                <option>Female</option>
            </select><br><br>

            Married:
            <select name="Married">
                <option>Yes</option>
                <option>No</option>
            </select><br><br>

            Dependents:
            <select name="Dependents">
                <option>0</option>
                <option>1</option>
                <option>2</option>
                <option>3+</option>
            </select><br><br>

            Education:
            <select name="Education">
                <option>Graduate</option>
                <option>Not Graduate</option>
            </select><br><br>

            Self Employed:
            <select name="Self_Employed">
                <option>No</option>
                <option>Yes</option>
            </select><br><br>

            Applicant Income:
            <input type="number" name="ApplicantIncome"
                   min="0" required><br><br>

            Coapplicant Income:
            <input type="number" name="CoapplicantIncome"
                   min="0" value="0" required><br><br>

            Loan Amount:
            <input type="number" name="LoanAmount"
                   min="0" required><br><br>

            Loan Amount Term:
            <input type="number" name="Loan_Amount_Term"
                   min="1" value="360" required><br><br>

            Credit History:
            <select name="Credit_History">
                <option value="1.0">Good</option>
                <option value="0.0">Poor</option>
            </select><br><br>

            Property Area:
            <select name="Property_Area">
                <option>Urban</option>
                <option>Semiurban</option>
                <option>Rural</option>
            </select><br><br>

            <button type="submit">Predict Loan Status</button>
        </form>
    </body>
    </html>
    """

@app.route("/predict", methods=["POST"])
def predict():
    try:
        input_data = {
            "Gender": request.form["Gender"],
            "Married": request.form["Married"],
            "Dependents": request.form["Dependents"],
            "Education": request.form["Education"],
            "Self_Employed": request.form["Self_Employed"],
            "ApplicantIncome": int(request.form["ApplicantIncome"]),
            "CoapplicantIncome": int(request.form["CoapplicantIncome"]),
            "LoanAmount": float(request.form["LoanAmount"]),
            "Loan_Amount_Term": float(
                request.form["Loan_Amount_Term"]
            ),
            "Credit_History": float(request.form["Credit_History"]),
            "Property_Area": request.form["Property_Area"]
        }

        input_df = pd.DataFrame([input_data])
        input_df = input_df.reindex(columns=X.columns)

        prediction = model.predict(input_df)[0]

        if str(prediction).strip().upper() == "Y":
            result = "Loan Approved"
        else:
            result = "Loan Not Approved"

        return f"""
        <html>
        <body style="font-family:Arial; margin:40px;">
            <h2>Prediction Result</h2>
            <h3>{result}</h3>
            <a href="/">Predict Again</a>
        </body>
        </html>
        """

    except Exception as e:
        return f"Prediction Error: {str(e)}", 400

print("Flask application is ready!")

Flask application is ready!


In [ ]:
from pyngrok import ngrok
from getpass import getpass
from threading import Thread
import time

token = getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(token)

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

Thread(target=run_flask, daemon=True).start()

time.sleep(2)

public_url = ngrok.connect(5000).public_url

print("Your Loan Prediction Website:")
print(public_url)

Enter your ngrok auth token: ··········
 * Serving Flask app 'loan_prediction'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Your Loan Prediction Website:
https://enlarged-pelican-alienate.ngrok-free.dev
